# BitLocker Status Analysis

This notebook provides interactive analysis and visualization of diagnostic data collected by `Get-BitLockerStatus.ps1`.

## Setup

1. Set `DATA_DIR` below to point to your collected logs folder (e.g., `BitLockerStatus-DD-MM-YYYY-HH-MM`).
2. Run all cells to generate the analysis report.

## Contents

- [Configuration](#Configuration)
- [BitLocker Volume Status](#BitLocker-Volume-Status)
- [MDM Policy Analysis](#MDM-Policy-Analysis)
- [Event Log Analysis](#Event-Log-Analysis)
- [Summary](#Summary)

## Configuration

In [ ]:
# Import required libraries
import os
import re
from pathlib import Path
from datetime import datetime
from typing import Optional

import pandas as pd
from lxml import etree

# Optional imports - graceful degradation if not available
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTTING = True
    sns.set_theme(style="whitegrid")
    plt.rcParams['figure.figsize'] = [12, 6]
except ImportError:
    HAS_PLOTTING = False
    print("Warning: matplotlib/seaborn not available. Visualizations will be skipped.")

try:
    from Evtx.Evtx import Evtx
    from Evtx.Views import evtx_file_xml_view
    HAS_EVTX = True
except ImportError:
    HAS_EVTX = False
    print("Warning: python-evtx not available. Event log parsing will be skipped.")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Set the path to your BitLocker diagnostic data folder
# Update this to point to the folder created by Get-BitLockerStatus.ps1
DATA_DIR = Path("./ExampleData")  # Change this to your actual data folder

# Validate the data directory exists
if not DATA_DIR.exists():
    print(f"ERROR: Data directory not found: {DATA_DIR}")
    print("Please update DATA_DIR to point to your BitLocker diagnostic folder.")
else:
    print(f"Data directory: {DATA_DIR.resolve()}")
    print(f"\nAvailable files:")
    for f in sorted(DATA_DIR.rglob("*")):
        if f.is_file():
            size_kb = f.stat().st_size / 1024
            print(f"  {f.relative_to(DATA_DIR)} ({size_kb:.1f} KB)")

## BitLocker Volume Status

Parse and display the output from `Get-BitLockerVolume`.

In [ ]:
def parse_bitlocker_volume(file_path: Path) -> Optional[pd.DataFrame]:
    """
    Parse Get-BitLockerVolume.txt output into a DataFrame.
    
    The output format is a series of key-value pairs for each volume,
    separated by blank lines.
    """
    if not file_path.exists():
        return None
    
    content = file_path.read_text(encoding='utf-8', errors='ignore')
    
    # Split into volume blocks (separated by double newlines or volume headers)
    volumes = []
    current_volume = {}
    
    for line in content.splitlines():
        line = line.strip()
        
        # Skip empty lines but use them to separate volumes
        if not line:
            if current_volume:
                volumes.append(current_volume)
                current_volume = {}
            continue
        
        # Parse key : value pairs
        if ':' in line:
            # Handle multi-colon lines (e.g., timestamps)
            parts = line.split(':', 1)
            if len(parts) == 2:
                key = parts[0].strip()
                value = parts[1].strip()
                
                # Handle special cases where value contains more colons
                if key and value:
                    current_volume[key] = value
                elif key and not value:
                    # Value might be on next lines (like KeyProtector)
                    current_volume[key] = ""
    
    # Don't forget the last volume
    if current_volume:
        volumes.append(current_volume)
    
    if not volumes:
        return None
    
    return pd.DataFrame(volumes)

In [ ]:
# Parse BitLocker volume status
bitlocker_file = DATA_DIR / "Get-BitLockerVolume.txt"
bitlocker_df = parse_bitlocker_volume(bitlocker_file)

if bitlocker_df is not None:
    print("BitLocker Volume Status")
    print("=" * 50)
    display(bitlocker_df)
    
    # Show key columns if they exist
    key_cols = ['MountPoint', 'VolumeStatus', 'ProtectionStatus', 'EncryptionMethod', 'EncryptionPercentage']
    available_cols = [c for c in key_cols if c in bitlocker_df.columns]
    if available_cols:
        print("\nKey Status Summary:")
        display(bitlocker_df[available_cols])
else:
    print(f"BitLocker volume file not found: {bitlocker_file}")
    print("This file is generated when Get-BitLockerStatus.ps1 runs successfully.")

## MDM Policy Analysis

Parse BitLocker MDM configuration from the MDM diagnostic report.

In [ ]:
def parse_mdm_bitlocker_xml(file_path: Path) -> Optional[pd.DataFrame]:
    """
    Parse BitlockerMDM.xml or extract BitLocker areas from MDMDiagReport.xml.
    
    Returns a DataFrame with policy settings.
    """
    if not file_path.exists():
        return None
    
    try:
        tree = etree.parse(str(file_path))
        root = tree.getroot()
        
        policies = []
        
        # Look for BitLocker policy areas in MDMDiagReport.xml format
        # Search for PolicyAreaName containing 'BitLocker'
        for area in root.iter():
            if area.tag == 'PolicyAreaName' and 'BitLocker' in (area.text or ''):
                # Get the parent element which should contain the policy details
                parent = area.getparent()
                if parent is not None:
                    policy_data = {'AreaName': area.text}
                    for child in parent:
                        if child.text and child.tag != 'PolicyAreaName':
                            policy_data[child.tag] = child.text
                    if len(policy_data) > 1:
                        policies.append(policy_data)
        
        # Also look for CSP-style BitLocker settings
        for elem in root.iter():
            elem_text = elem.text or ''
            if 'BitLocker' in elem.tag or 'BitLocker' in elem_text:
                if elem.tag not in ['PolicyAreaName']:
                    policy_entry = {
                        'Setting': elem.tag,
                        'Value': elem_text[:200] if elem_text else 'N/A',
                        'Path': '/'.join([p.tag for p in elem.iterancestors()][:3][::-1])
                    }
                    # Avoid duplicates
                    if policy_entry not in policies:
                        policies.append(policy_entry)
        
        if policies:
            return pd.DataFrame(policies)
        return None
        
    except etree.XMLSyntaxError as e:
        print(f"XML parsing error: {e}")
        return None

In [ ]:
def extract_bitlocker_csp_policies(file_path: Path) -> Optional[pd.DataFrame]:
    """
    Extract BitLocker CSP policies from MDMDiagReport.xml.
    
    Looks for specific BitLocker configuration nodes in the MDM report.
    """
    if not file_path.exists():
        return None
    
    try:
        tree = etree.parse(str(file_path))
        root = tree.getroot()
        
        # Known BitLocker policy settings to look for
        bitlocker_settings = [
            'RequireDeviceEncryption',
            'AllowWarningForOtherDiskEncryption',
            'EncryptionMethodByDriveType',
            'SystemDrivesRequireStartupAuthentication',
            'SystemDrivesMinimumPINLength',
            'SystemDrivesRecoveryMessage',
            'SystemDrivesRecoveryOptions',
            'FixedDrivesRecoveryOptions',
            'FixedDrivesRequireEncryption',
            'RemovableDrivesRequireEncryption',
        ]
        
        results = []
        
        # Search for elements containing BitLocker settings
        for setting in bitlocker_settings:
            for elem in root.iter():
                if setting in elem.tag or (elem.text and setting in elem.text):
                    results.append({
                        'Setting': setting,
                        'Element': elem.tag,
                        'Value': (elem.text or '')[:100]
                    })
        
        # Find configuration areas with "BitLocker" in policy area name
        for area in root.iter('PolicyAreaName'):
            if area.text and 'BitLocker' in area.text:
                parent = area.getparent()
                if parent is not None:
                    for child in parent:
                        if child.tag != 'PolicyAreaName' and child.text:
                            results.append({
                                'Setting': child.tag,
                                'Element': 'PolicyArea',
                                'Value': child.text[:100]
                            })
        
        if results:
            df = pd.DataFrame(results).drop_duplicates()
            return df
        return None
        
    except Exception as e:
        print(f"Error parsing MDM report: {e}")
        return None

In [ ]:
# Try to find MDM BitLocker configuration
mdm_files = [
    DATA_DIR / "MDM" / "BitlockerMDM.xml",  # Extracted by Get-BitLockerStatus.ps1
    DATA_DIR / "BitlockerMDM.xml",           # Alternative location
    DATA_DIR / "MDMDiagReport.xml",          # Full MDM report
]

mdm_df = None
mdm_source = None

for mdm_file in mdm_files:
    if mdm_file.exists():
        print(f"Found MDM data: {mdm_file}")
        mdm_source = mdm_file
        
        # Try specialized extraction first
        mdm_df = extract_bitlocker_csp_policies(mdm_file)
        if mdm_df is None:
            mdm_df = parse_mdm_bitlocker_xml(mdm_file)
        
        if mdm_df is not None:
            break

if mdm_df is not None:
    print(f"\nBitLocker MDM Policy Settings (from {mdm_source.name})")
    print("=" * 60)
    display(mdm_df)
else:
    print("No MDM BitLocker configuration found.")
    print("MDM data is only present if Get-BitLockerStatus.ps1 was run with -MDM flag.")

## Event Log Analysis

Parse Windows Event Logs (.evtx) for BitLocker-related events.

In [ ]:
def parse_evtx_file(file_path: Path, max_events: int = 1000) -> Optional[pd.DataFrame]:
    """
    Parse a Windows Event Log (.evtx) file into a DataFrame.
    
    Args:
        file_path: Path to the .evtx file
        max_events: Maximum number of events to parse (for performance)
    
    Returns:
        DataFrame with event data or None if parsing fails
    """
    if not HAS_EVTX:
        print("python-evtx library not available. Install with: pip install python-evtx")
        return None
    
    if not file_path.exists():
        return None
    
    events = []
    
    try:
        with Evtx(str(file_path)) as evtx:
            for i, record in enumerate(evtx.records()):
                if i >= max_events:
                    break
                
                try:
                    xml_str = record.xml()
                    root = etree.fromstring(xml_str.encode('utf-8'))
                    
                    # Extract common event fields
                    ns = {'e': 'http://schemas.microsoft.com/win/2004/08/events/event'}
                    
                    system = root.find('e:System', ns)
                    event_data = root.find('e:EventData', ns)
                    
                    event = {
                        'EventID': system.findtext('e:EventID', default='', namespaces=ns),
                        'TimeCreated': '',
                        'Level': system.findtext('e:Level', default='', namespaces=ns),
                        'Provider': '',
                        'Message': ''
                    }
                    
                    # Get timestamp
                    time_elem = system.find('e:TimeCreated', ns)
                    if time_elem is not None:
                        event['TimeCreated'] = time_elem.get('SystemTime', '')
                    
                    # Get provider
                    provider_elem = system.find('e:Provider', ns)
                    if provider_elem is not None:
                        event['Provider'] = provider_elem.get('Name', '')
                    
                    # Get event data/message
                    if event_data is not None:
                        data_items = []
                        for data in event_data:
                            name = data.get('Name', '')
                            value = data.text or ''
                            if name:
                                data_items.append(f"{name}={value}")
                            elif value:
                                data_items.append(value)
                        event['Message'] = '; '.join(data_items)[:500]
                    
                    events.append(event)
                    
                except Exception as e:
                    continue
        
        if events:
            df = pd.DataFrame(events)
            # Convert timestamp to datetime
            if 'TimeCreated' in df.columns:
                df['TimeCreated'] = pd.to_datetime(df['TimeCreated'], errors='coerce')
                df = df.sort_values('TimeCreated', ascending=False)
            return df
        
        return None
        
    except Exception as e:
        print(f"Error parsing {file_path}: {e}")
        return None

In [ ]:
# Parse BitLocker API event log
bitlocker_evtx = DATA_DIR / "Microsoft-Windows-BitLocker-API_Management.evtx"
bitlocker_events_df = parse_evtx_file(bitlocker_evtx)

if bitlocker_events_df is not None:
    print(f"BitLocker API Events ({len(bitlocker_events_df)} events)")
    print("=" * 60)
    display(bitlocker_events_df.head(20))
    
    # Show event ID distribution
    print("\nEvent ID Distribution:")
    display(bitlocker_events_df['EventID'].value_counts().head(10))
else:
    print(f"BitLocker event log not found: {bitlocker_evtx}")

In [ ]:
# Parse System event log (for TPM and boot-related events)
system_evtx = DATA_DIR / "system.evtx"
system_events_df = parse_evtx_file(system_evtx, max_events=500)

if system_events_df is not None:
    # Filter for potentially relevant events (TPM, BitLocker, boot)
    relevant_providers = ['Microsoft-Windows-TPM', 'Microsoft-Windows-BitLocker', 'Microsoft-Windows-Kernel']
    
    # Show sample of system events
    print(f"System Events ({len(system_events_df)} events parsed)")
    print("=" * 60)
    display(system_events_df.head(10))
    
    # Provider distribution
    print("\nTop Event Providers:")
    display(system_events_df['Provider'].value_counts().head(10))
else:
    print(f"System event log not found: {system_evtx}")

In [ ]:
# Visualize event timeline (if matplotlib available)
if HAS_PLOTTING and bitlocker_events_df is not None and 'TimeCreated' in bitlocker_events_df.columns:
    # Filter valid timestamps
    events_with_time = bitlocker_events_df.dropna(subset=['TimeCreated'])
    
    if len(events_with_time) > 0:
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # Group by hour and count events
        events_with_time['Hour'] = events_with_time['TimeCreated'].dt.floor('H')
        hourly_counts = events_with_time.groupby('Hour').size()
        
        if len(hourly_counts) > 1:
            hourly_counts.plot(kind='bar', ax=ax, color='steelblue', alpha=0.7)
            ax.set_xlabel('Time')
            ax.set_ylabel('Event Count')
            ax.set_title('BitLocker Events Over Time')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
        else:
            print("Not enough time-distributed events for timeline visualization.")
    else:
        print("No events with valid timestamps for visualization.")

## Summary

Consolidated findings from all data sources.

In [ ]:
def generate_summary():
    """
    Generate a consolidated summary of all parsed data.
    """
    summary = {
        'Data Source': [],
        'Status': [],
        'Details': []
    }
    
    # BitLocker Volume Status
    if bitlocker_df is not None:
        vol_count = len(bitlocker_df)
        status = 'OK' if vol_count > 0 else 'Warning'
        details = f"{vol_count} volume(s) found"
        
        # Check protection status if available
        if 'ProtectionStatus' in bitlocker_df.columns:
            protected = bitlocker_df['ProtectionStatus'].str.contains('On', case=False, na=False).sum()
            details += f", {protected} protected"
    else:
        status = 'Missing'
        details = 'Get-BitLockerVolume.txt not found'
    
    summary['Data Source'].append('BitLocker Volumes')
    summary['Status'].append(status)
    summary['Details'].append(details)
    
    # MDM Policy
    if mdm_df is not None:
        policy_count = len(mdm_df)
        summary['Data Source'].append('MDM Policies')
        summary['Status'].append('OK')
        summary['Details'].append(f"{policy_count} BitLocker policy setting(s) found")
    else:
        summary['Data Source'].append('MDM Policies')
        summary['Status'].append('N/A')
        summary['Details'].append('No MDM data (requires -MDM flag during collection)')
    
    # BitLocker Events
    if bitlocker_events_df is not None:
        event_count = len(bitlocker_events_df)
        summary['Data Source'].append('BitLocker Events')
        summary['Status'].append('OK')
        summary['Details'].append(f"{event_count} event(s) in log")
    else:
        summary['Data Source'].append('BitLocker Events')
        summary['Status'].append('Missing')
        summary['Details'].append('Event log file not found')
    
    # System Events
    if system_events_df is not None:
        event_count = len(system_events_df)
        summary['Data Source'].append('System Events')
        summary['Status'].append('OK')
        summary['Details'].append(f"{event_count} event(s) parsed")
    else:
        summary['Data Source'].append('System Events')
        summary['Status'].append('Missing')
        summary['Details'].append('System event log not found')
    
    return pd.DataFrame(summary)

In [ ]:
print("\n" + "=" * 60)
print("ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nData Directory: {DATA_DIR.resolve()}")
print(f"Analysis Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()

summary_df = generate_summary()
display(summary_df)

# Status indicators
missing_count = (summary_df['Status'] == 'Missing').sum()
if missing_count > 0:
    print(f"\n[!] {missing_count} data source(s) missing - some analyses may be incomplete.")
else:
    print("\n[OK] All expected data sources are present.")

---

## Next Steps

Based on the analysis above, consider:

1. **If BitLocker is not enabled**: Review MDM policies and Group Policy settings (FVE_Policies.reg)
2. **If encryption failed**: Check event logs for error codes and TPM status (Get-TPM.txt)
3. **For key escrow issues**: Verify MDM enrollment status and Azure AD connectivity
4. **For recovery scenarios**: Document the recovery key protectors shown in BitLocker volume output

For additional analysis, you can modify the cells above or add new cells to explore specific aspects of the collected data.